In [0]:
Select *
FROM `brightlearn`.`default`.`fnb_sales_case_study_2021_1`;

SELECT 
  ROUND(AVG(`Sales`), 2) AS `Avg_Daily_Sales`,
  ROUND(AVG(`Quantity Sold`), 0) AS `Avg_Daily_Quantity`
FROM `brightlearn`.`default`.`fnb_sales_case_study_2021_1`;

SELECT
  `Date`,
  `Sales`,
  `Cost Of Sales`,
  `Quantity Sold`,
  Dayname(`Date`) as Dayname,
  Monthname(`Date`) AS Monthname,
  Case
  When DAYOFWEEK(`Date`) IN (1,7) Then 'Weekend'
  Else 'Weekday'
  End as Day_type,
  ROUND(`Sales` / `Quantity Sold`, 2) AS `Price_Per_Unit`,
  `Sales` - `Cost Of Sales` AS `Gross_Profit`,
  ROUND(((`Sales` - `Cost Of Sales`) / `Sales`) * 100, 2) AS `Gross_Profit_Percentage`,
  ROUND(`Sales` / `Quantity Sold`, 2) AS `Sales_Price_Per_Unit`,
  ROUND(`Cost Of Sales` / `Quantity Sold`, 2) AS `Cost_Per_Unit`,
  ROUND((`Sales` / `Quantity Sold`) - (`Cost Of Sales` / `Quantity Sold`), 2) AS `Gross_Profit_Per_Unit`,
  ROUND((((`Sales` / `Quantity Sold`) - (`Cost Of Sales` / `Quantity Sold`)) / (`Sales` / `Quantity Sold`)) * 100, 2) AS `Gross_Profit_Percentage_Per_Unit`
FROM `brightlearn`.`default`.`fnb_sales_case_study_2021_1`
WHERE `Sales` > 0 AND `Quantity Sold` > 0
ORDER BY `Date`

WITH monthly_data AS (
  SELECT
    DATE_TRUNC('month', `Date`) AS `Month`,
    SUM(`Sales` - `Cost Of Sales`) AS Gross_Profit,
    ROUND(
AVG((`Sales`-`Cost Of Sales`)/`Sales`*100),2) AS Avg_Margin_Pct,
    ROUND(AVG(`Sales` / `Quantity Sold`), 2) AS `Avg_Price_Per_Unit`,
    ROUND(SUM(`Quantity Sold`), 0) AS `Total_Quantity_Sold`
  FROM `brightlearn`.`default`.`fnb_sales_case_study_2021_1`
  WHERE `Quantity Sold` > 0
  GROUP BY DATE_TRUNC('month', `Date`)
),
lagged_data AS (
  SELECT
    `Month`,
    `Avg_Price_Per_Unit`,
    `Total_Quantity_Sold`,
    LAG(`Avg_Price_Per_Unit`) OVER (ORDER BY `Month`) AS `Prev_Price`,
    LAG(`Total_Quantity_Sold`) OVER (ORDER BY `Month`) AS `Prev_Quantity`
  FROM monthly_data
),
elasticity_data AS (
  SELECT 
    `Month`,
    `Avg_Price_Per_Unit` AS `Current_Price`,
    `Prev_Price`,
    `Total_Quantity_Sold` AS `Current_Quantity`,
    `Prev_Quantity`,
    ROUND(((`Avg_Price_Per_Unit` - `Prev_Price`) / `Prev_Price`) * 100, 2) AS `Price_Change_Pct`,
    ROUND(((`Total_Quantity_Sold` - `Prev_Quantity`) / `Prev_Quantity`) * 100, 2) AS `Quantity_Change_Pct`,
    ROUND(
      ((`Total_Quantity_Sold` - `Prev_Quantity`) / `Prev_Quantity`) / 
      ((`Avg_Price_Per_Unit` - `Prev_Price`) / `Prev_Price`), 
      2
    ) AS `Price_Elasticity_of_Demand`
  FROM lagged_data
  WHERE `Prev_Price` IS NOT NULL 
    AND `Prev_Quantity` IS NOT NULL
    AND (`Avg_Price_Per_Unit` - `Prev_Price`) != 0
)
SELECT
  `Month`,
  `Current_Price`,
  `Prev_Price`,
  `Current_Quantity`,
  `Prev_Quantity`,
  `Price_Change_Pct`,
  `Quantity_Change_Pct`,
  `Price_Elasticity_of_Demand`,
  CASE
    WHEN ABS(`Price_Elasticity_of_Demand`) > 1 THEN 'Elastic Demand'
    WHEN ABS(`Price_Elasticity_of_Demand`) < 1 THEN 'Inelastic Demand'
    ELSE 'Unit Elastic'
  END AS `Demand_Type`
FROM elasticity_data
ORDER BY `Month`
limit 100;

